In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
OUTPUTS_DIR = Path("../Outputs")

# Emotion model outputs
emo_distil = pd.read_csv(OUTPUTS_DIR / "emotion_distilroberta_905.csv")
emo_goemo = pd.read_csv(OUTPUTS_DIR / "emotion_goemotions_905.csv")
emo_cardiff = pd.read_csv(OUTPUTS_DIR / "emotion_cardiff_twitter_905.csv")

# Toxicity model outputs
tox_detox = pd.read_csv(OUTPUTS_DIR / "toxicity_detoxify_905.csv")
tox_snlp = pd.read_csv(OUTPUTS_DIR / "toxicity_sNLP_905.csv")
tox_cardiff = pd.read_csv(OUTPUTS_DIR / "toxicity_cardiff_offensive_905.csv")

print(f"DistilRoBERTa emotion: {emo_distil.shape}")
print(f"GoEmotions emotion:    {emo_goemo.shape}")
print(f"Cardiff Twitter emotion: {emo_cardiff.shape}")
print(f"Detoxify toxicity:     {tox_detox.shape}")
print(f"s-nlp toxicity:        {tox_snlp.shape}")
print(f"Cardiff Offensive:     {tox_cardiff.shape}")

DistilRoBERTa emotion: (905, 31)
GoEmotions emotion:    (905, 52)
Cardiff Twitter emotion: (905, 35)
Detoxify toxicity:     (905, 31)
s-nlp toxicity:        (905, 26)
Cardiff Offensive:     (905, 26)


In [3]:
print("=== DistilRoBERTa columns ===")
print([c for c in emo_distil.columns if c.startswith("emo_") or c == "dominant_emotion"])

print("\n=== GoEmotions columns ===")
print([c for c in emo_goemo.columns if c.startswith("emo_") or c == "dominant_emotion"])

print("\n=== Cardiff Twitter columns ===")
print([c for c in emo_cardiff.columns if c.startswith("emo_") or c == "dominant_emotion"])

print("\n=== Detoxify columns ===")
print([c for c in tox_detox.columns if c.startswith("tox_") or "toxic" in c.lower()])

print("\n=== s-nlp columns ===")
print([c for c in tox_snlp.columns if "snlp" in c.lower()])

print("\n=== Cardiff Offensive columns ===")
print([c for c in tox_cardiff.columns if "cardiff" in c.lower()])

=== DistilRoBERTa columns ===
['emo_anger', 'emo_disgust', 'emo_fear', 'emo_joy', 'emo_neutral', 'emo_sadness', 'emo_surprise', 'dominant_emotion']

=== GoEmotions columns ===
['emo_admiration', 'emo_amusement', 'emo_anger', 'emo_annoyance', 'emo_approval', 'emo_caring', 'emo_confusion', 'emo_curiosity', 'emo_desire', 'emo_disappointment', 'emo_disapproval', 'emo_disgust', 'emo_embarrassment', 'emo_excitement', 'emo_fear', 'emo_gratitude', 'emo_grief', 'emo_joy', 'emo_love', 'emo_nervousness', 'emo_neutral', 'emo_optimism', 'emo_pride', 'emo_realization', 'emo_relief', 'emo_remorse', 'emo_sadness', 'emo_surprise', 'dominant_emotion']

=== Cardiff Twitter columns ===
['emo_anger', 'emo_anticipation', 'emo_disgust', 'emo_fear', 'emo_joy', 'emo_love', 'emo_optimism', 'emo_pessimism', 'emo_sadness', 'emo_surprise', 'emo_trust', 'dominant_emotion']

=== Detoxify columns ===
['tox_toxicity', 'tox_severe_toxicity', 'tox_obscene', 'tox_identity_attack', 'tox_insult', 'tox_threat', 'is_toxic', 

In [13]:
# Start with the merged_text file as the base (has student_id, text_source, has_text, text_for_analysis)
merged = pd.read_csv(OUTPUTS_DIR / "merged_text_905.csv")

# Keep only the columns we need from the base
master = merged[["student_id", "text_source", "has_text", "text_for_analysis"]].copy()
master["post_index"] = master.index

print(f"Master dataframe: {master.shape}")
print(f"Posts with text: {master['has_text'].sum()}")
master.head()

Master dataframe: (905, 5)
Posts with text: 797


,student_id,text_source,has_text,text_for_analysis,post_index
0,1_A,caption+transcript,True,Chinese authorities say their investigation of...,0
1,1_A,caption+transcript,True,You might be feeling overwhelmed with everythi...,1
2,1_A,caption_only,True,Super Bowl Monday is a holiday next year ??,2
3,2_A,caption_only,True,Explore D.Desirable at lON OrchardSingapore,3
4,2_A,caption_only,True,i-dle OFFICIAL LIGHT STICK VER.3 COMING SOON,4


In [14]:
# Tighter mapping: only combine very close synonyms
# Format: { our_emotion: [list of source labels from that model] }

DISTIL_MAP = {
    "anger": ["emo_anger"],
    "disgust": ["emo_disgust"],
    "fear": ["emo_fear"],
    "joy": ["emo_joy"],
    "sadness": ["emo_sadness"],
    "surprise": ["emo_surprise"],
}

GOEMO_MAP = {
    "anger": ["emo_anger", "emo_annoyance"],
    "disgust": ["emo_disgust"],
    "fear": ["emo_fear", "emo_nervousness"],
    "joy": ["emo_joy", "emo_amusement", "emo_excitement"],
    "sadness": ["emo_sadness", "emo_disappointment", "emo_grief"],
    "surprise": ["emo_surprise", "emo_realization"],
}

CARDIFF_MAP = {
    "anger": ["emo_anger"],
    "disgust": ["emo_disgust"],
    "fear": ["emo_fear"],
    "joy": ["emo_joy"],
    "sadness": ["emo_sadness"],
    "surprise": ["emo_surprise"],
}

LOCKED_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

print("Tighter mapping defined for 3 emotion models")
print(f"DistilRoBERTa: {sum(len(v) for v in DISTIL_MAP.values())} source labels → 6")
print(f"GoEmotions:    {sum(len(v) for v in GOEMO_MAP.values())} source labels → 6")
print(f"Cardiff:       {sum(len(v) for v in CARDIFF_MAP.values())} source labels → 6")

Tighter mapping defined for 3 emotion models
DistilRoBERTa: 6 source labels → 6
GoEmotions:    13 source labels → 6
Cardiff:       6 source labels → 6


In [15]:
def collapse_emotions(df, mapping, model_prefix):
    """For each post, combine source labels into 6 emotion scores using MAX (generous)."""
    result = pd.DataFrame(index=df.index)
    for our_emo, source_labels in mapping.items():
        # Only use source labels that actually exist in the dataframe
        valid = [c for c in source_labels if c in df.columns]
        if len(valid) == 0:
            result[f"{model_prefix}_{our_emo}"] = 0.0
        else:
            # Generous = take the MAX across related labels
            result[f"{model_prefix}_{our_emo}"] = df[valid].max(axis=1)
    return result

distil_emo = collapse_emotions(emo_distil, DISTIL_MAP, "distil")
goemo_emo = collapse_emotions(emo_goemo, GOEMO_MAP, "goemo")
cardiff_emo = collapse_emotions(emo_cardiff, CARDIFF_MAP, "cardiff")

# Join all to master
master = master.join(distil_emo).join(goemo_emo).join(cardiff_emo)

print(f"Master shape after emotion mapping: {master.shape}")
print("\nSample row (first post with text):")
sample = master[master["has_text"]].iloc[0]
for emo in LOCKED_EMOTIONS:
    print(f"  {emo:10s}  distil={sample[f'distil_{emo}']:.2f}  goemo={sample[f'goemo_{emo}']:.2f}  cardiff={sample[f'cardiff_{emo}']:.2f}")

Master shape after emotion mapping: (905, 23)

Sample row (first post with text):
  anger       distil=0.86  goemo=0.04  cardiff=0.77
  disgust     distil=0.00  goemo=0.01  cardiff=0.82
  fear        distil=0.09  goemo=0.00  cardiff=0.47
  joy         distil=0.01  goemo=0.00  cardiff=0.01
  sadness     distil=0.03  goemo=0.04  cardiff=0.23
  surprise    distil=0.00  goemo=0.07  cardiff=0.13


In [16]:
for emo in LOCKED_EMOTIONS:
    cols = [f"distil_{emo}", f"goemo_{emo}", f"cardiff_{emo}"]
    master[f"emo_{emo}_max"] = master[cols].max(axis=1)
    master[f"emo_{emo}_min"] = master[cols].min(axis=1)
    master[f"emo_{emo}_range"] = master[f"emo_{emo}_max"] - master[f"emo_{emo}_min"]

# Overall emotion agreement: max range across all 6 emotions
range_cols = [f"emo_{e}_range" for e in LOCKED_EMOTIONS]
master["emo_max_range"] = master[range_cols].max(axis=1)

print("Agreement metrics computed per emotion")
print(f"\nAverage range per emotion (across all posts with text):")
for emo in LOCKED_EMOTIONS:
    avg = master[master["has_text"]][f"emo_{emo}_range"].mean()
    print(f"  {emo:10s}  avg range = {avg:.3f}")

print(f"\nDistribution of emo_max_range (overall disagreement):")
print(master[master["has_text"]]["emo_max_range"].describe())

Agreement metrics computed per emotion

Average range per emotion (across all posts with text):
  anger       avg range = 0.140
  disgust     avg range = 0.139
  fear        avg range = 0.198
  joy         avg range = 0.528
  sadness     avg range = 0.149
  surprise    avg range = 0.153

Distribution of emo_max_range (overall disagreement):
count    797.000000
mean       0.694197
std        0.246380
min        0.096326
25%        0.515181
50%        0.771308
75%        0.898702
max        0.988469
Name: emo_max_range, dtype: float64


In [17]:
# Detoxify subtypes
tox_cols = ["tox_toxicity", "tox_severe_toxicity", "tox_obscene", "tox_identity_attack", "tox_insult", "tox_threat"]
detox_subset = tox_detox[tox_cols].copy()
detox_subset.columns = [f"detox_{c.replace('tox_', '')}" for c in tox_cols]

# s-nlp binary
snlp_subset = tox_snlp[["snlp_toxic"]].copy()

# Cardiff offensive
cardiff_offensive_subset = tox_cardiff[["cardiff_offensive"]].copy()

master = master.join(detox_subset).join(snlp_subset).join(cardiff_offensive_subset)

# Compute the "broad toxicity" agreement across all 3 models (using detox_toxicity as the comparable Detoxify signal)
master["tox_max"] = master[["detox_toxicity", "snlp_toxic", "cardiff_offensive"]].max(axis=1)
master["tox_min"] = master[["detox_toxicity", "snlp_toxic", "cardiff_offensive"]].min(axis=1)
master["tox_range"] = master["tox_max"] - master["tox_min"]

print(f"Master shape after toxicity merge: {master.shape}")
print(f"\nToxicity range distribution (posts with text):")
print(master[master["has_text"]]["tox_range"].describe())

Master shape after toxicity merge: (905, 53)

Toxicity range distribution (posts with text):
count    797.000000
mean       0.148275
std        0.116027
min        0.021584
25%        0.072783
50%        0.114251
75%        0.185662
max        0.877342
Name: tox_range, dtype: float64


In [18]:
RANGE_THRESHOLD = 0.25  # the "moderate" agreement threshold you picked

# --- Emotion routing ---
def emotion_bucket(row):
    if not row["has_text"]:
        return "gray"
    # Text too short check
    text = str(row["text_for_analysis"])
    if len(text.strip()) < 20:
        return "gray"
    # If max disagreement across all emotions > threshold → Yellow
    if row["emo_max_range"] > RANGE_THRESHOLD:
        return "yellow"
    return "green"

# --- Toxicity routing ---
def toxicity_bucket(row):
    if not row["has_text"]:
        return "gray"
    text = str(row["text_for_analysis"])
    if len(text.strip()) < 20:
        return "gray"
    # Red flag triggers: high threat, identity attack, or severe toxicity
    if row["detox_threat"] > 0.5 or row["detox_identity_attack"] > 0.5 or row["detox_severe_toxicity"] > 0.5:
        return "red"
    # If broad models disagree significantly → Yellow
    if row["tox_range"] > RANGE_THRESHOLD:
        return "yellow"
    # If detox subtype is high but broad models are low → Yellow (model mismatch)
    if row["detox_insult"] > 0.5 and row["snlp_toxic"] < 0.3:
        return "yellow"
    return "green"

master["emotion_bucket"] = master.apply(emotion_bucket, axis=1)
master["toxicity_bucket"] = master.apply(toxicity_bucket, axis=1)

print("=== Emotion bucket counts ===")
print(master["emotion_bucket"].value_counts())
print("\n=== Toxicity bucket counts ===")
print(master["toxicity_bucket"].value_counts())

=== Emotion bucket counts ===
emotion_bucket
yellow    696
gray      156
green      53
Name: count, dtype: int64

=== Toxicity bucket counts ===
toxicity_bucket
green     665
gray      156
yellow     83
red         1
Name: count, dtype: int64


In [19]:
# For each post, find which model gave the score furthest from the OTHER two
# If one model is consistently the outlier, it's a red flag

def find_outlier_model(row, emotion):
    """Returns which model's score is the outlier for a given emotion on this post."""
    scores = {
        "distil": row[f"distil_{emotion}"],
        "goemo": row[f"goemo_{emotion}"],
        "cardiff": row[f"cardiff_{emotion}"]
    }
    # Compute each model's distance from the average of the other two
    distances = {}
    for model in scores:
        others = [scores[m] for m in scores if m != model]
        avg_others = np.mean(others)
        distances[model] = abs(scores[model] - avg_others)
    # Return the model with the largest distance (the outlier)
    return max(distances, key=distances.get)

# Run on yellow-bucket posts (where there was real disagreement)
yellow_subset = master[master["emotion_bucket"] == "yellow"].copy()

outlier_counts = {"distil": 0, "goemo": 0, "cardiff": 0}
for _, row in yellow_subset.iterrows():
    # Find which emotion had the biggest range for this post
    ranges = {e: row[f"emo_{e}_range"] for e in LOCKED_EMOTIONS}
    biggest_emo = max(ranges, key=ranges.get)
    # Find the outlier model for that emotion
    outlier = find_outlier_model(row, biggest_emo)
    outlier_counts[outlier] += 1

total_yellow = len(yellow_subset)
print(f"=== Outlier model frequency in {total_yellow} yellow-bucket posts ===")
for model, count in outlier_counts.items():
    pct = (count / total_yellow * 100) if total_yellow > 0 else 0
    flag = " ⚠️ RED FLAG" if pct > 50 else ""
    print(f"  {model:10s}  {count:4d} posts ({pct:.1f}%){flag}")

print("\nNote: A model is flagged as 'consistently disagreeing' if it's the outlier in >50% of yellow posts.")

=== Outlier model frequency in 696 yellow-bucket posts ===
  distil       116 posts (16.7%)
  goemo        193 posts (27.7%)
  cardiff      387 posts (55.6%) ⚠️ RED FLAG

Note: A model is flagged as 'consistently disagreeing' if it's the outlier in >50% of yellow posts.


In [20]:
# Show 3 yellow-bucket posts with all model scores
yellow_sample = master[master["emotion_bucket"] == "yellow"].head(3)
for _, row in yellow_sample.iterrows():
    print(f"\n=== Post {row['post_index']} [{row['text_source']}] ===")
    print(f"Text: {str(row['text_for_analysis'])[:200]}")
    print("\nModel scores per emotion:")
    for emo in LOCKED_EMOTIONS:
        d = row[f"distil_{emo}"]
        g = row[f"goemo_{emo}"]
        c = row[f"cardiff_{emo}"]
        r = row[f"emo_{emo}_range"]
        flag = " ←" if r > 0.25 else ""
        print(f"  {emo:10s}  distil={d:.2f}  goemo={g:.2f}  cardiff={c:.2f}  range={r:.2f}{flag}")


=== Post 0 [caption+transcript] ===
Text: Chinese authorities say their investigation of a high-profile scandal at one of the countrys leading state-run museums has revealed systemic mismanagement and alleged corruption over decades. #nanjin

Model scores per emotion:
  anger       distil=0.86  goemo=0.04  cardiff=0.77  range=0.82 ←
  disgust     distil=0.00  goemo=0.01  cardiff=0.82  range=0.82 ←
  fear        distil=0.09  goemo=0.00  cardiff=0.47  range=0.47 ←
  joy         distil=0.01  goemo=0.00  cardiff=0.01  range=0.01
  sadness     distil=0.03  goemo=0.04  cardiff=0.23  range=0.20
  surprise    distil=0.00  goemo=0.07  cardiff=0.13  range=0.13

=== Post 1 [caption+transcript] ===
Text: You might be feeling overwhelmed with everything thats going on lately
.
Reframing your stress like this is called a stress is enhancing mindset and there is a lot of research that shows your body p

Model scores per emotion:
  anger       distil=0.02  goemo=0.01  cardiff=0.04  range=0.03
  di

# Routing Pipeline (905 dataset)

This notebook implements the confidence routing pipeline for emotion and toxicity classification across the 905-post student dataset.

## Pipeline Overview

For each post, the workflow is:

1. Load outputs from all 6 transformer models (3 emotion + 3 toxicity)
2. Map each emotion model's labels into the 6 locked emotions (anger, disgust, fear, joy, sadness, surprise)
3. Compute per-emotion agreement metrics across the 3 emotion models (max, min, range)
4. Compute toxicity agreement across Detoxify, s-nlp, and Cardiff Offensive
5. Route each post into a confidence bucket:
   - **Green**: high agreement → use transformer scores directly
   - **Yellow**: model disagreement → route to LLM for audit
   - **Red**: sensitive flags (threat, identity attack, severe toxicity) → LLM + human review
   - **Gray**: insufficient context (no text or under 20 characters) → drop
6. Check for "consistently disagreeing" models as a methodology red flag

## Methodology Notes

**Emotion label mapping (tighter version used):**
- DistilRoBERTa (7 labels) → 6: one-to-one mapping (dropped "neutral")
- GoEmotions (28 labels) → 6: collapsed only close synonyms
  - anger ← anger, annoyance
  - fear ← fear, nervousness
  - joy ← joy, amusement, excitement
  - sadness ← sadness, disappointment, grief
  - surprise ← surprise, realization
- Cardiff Twitter (11 labels) → 6: one-to-one mapping (dropped love, optimism, pessimism, trust, anticipation)

The earlier "generous" mapping inflated joy scores by combining 11 GoEmotions labels into joy and 5 Cardiff labels into joy. Switching to a tighter mapping changed the bucket distribution by less than 2%, confirming the disagreements are real and not artifacts of label mapping.

**Routing threshold:** 0.25 max range across models (moderate setting).

## Bucket Distribution

**Emotion:**
- Yellow: 696 (87%)
- Gray: 156 (20%)
- Green: 53 (7%)

**Toxicity:**
- Green: 665 (83%)
- Yellow: 83 (10%)
- Gray: 156 (20%)
- Red: 1

## Key Finding: Cardiff as the "Outlier"

Cardiff was the outlier model in 55.6% of yellow-bucket emotion posts. Initially this looks like a red flag, but manual inspection of sample posts shows Cardiff is more sensitive to social media tone (correctly detecting hype, excitement, and frustration in tweet-style content) while DistilRoBERTa and GoEmotions tend to under-fire on entertainment, news, and casual social media content.

Example: A post saying *"Super Bowl Monday is a holiday next year"* gets:
- DistilRoBERTa: 0.01 joy
- GoEmotions: 0.02 joy
- Cardiff: 0.75 joy

Cardiff correctly identifies the celebratory tone here. The "outlier" status isn't evidence of a bad model — it's evidence the other two miss subtle social-media signals.

## What Goes to LLM

Yellow + Red bucket = ~700 posts × 2 prompts (objective + youth) per task = ~2800 API calls. Estimated cost on Claude Sonnet 4.6 with batch API: ~$8-12 for emotion + toxicity scoring.

## Next Steps

1. Run LLM on Yellow + Red bucket posts (objective + youth prompts)
2. Apply confidence gating:
   - LLM confidence ≥ 0.8 → accept LLM result
   - LLM confidence 0.2 to 0.8 → flag for human review
   - LLM confidence < 0.2 → drop the post
3. Produce final consolidated output with per-post scores + confidence tiers
4. Generate flow diagram for the grant document

In [21]:
output_path = OUTPUTS_DIR / "routed_master_905.csv"
master.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {master.shape}")
print(f"\nColumns: {master.shape[1]}")
print("Bucket distribution:")
print(f"  Emotion: {dict(master['emotion_bucket'].value_counts())}")
print(f"  Toxicity: {dict(master['toxicity_bucket'].value_counts())}")

Saved: ../Outputs/routed_master_905.csv
Shape: (905, 55)

Columns: 55
Bucket distribution:
  Emotion: {'yellow': np.int64(696), 'gray': np.int64(156), 'green': np.int64(53)}
  Toxicity: {'green': np.int64(665), 'gray': np.int64(156), 'yellow': np.int64(83), 'red': np.int64(1)}
